<a href="https://colab.research.google.com/github/vishaljoshi24/DungeonsAndDragonsTurnClassification/blob/Codex-Branch/prompt_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone -b Codex-Branch https://github.com/vishaljoshi24/DungeonsAndDragonsTurnClassification/
%cd /content/DungeonsAndDragonsTurnClassification

In [ ]:
import requests

response = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)
print(response.status_code)
print(response.json())

In [ ]:
import subprocess

result = subprocess.run(
    ["ollama", "pull", "qwen3:4b"],
    capture_output=False,
    text=True,
)

print("Download finished with exit code:", result.returncode)

In [ ]:
import requests

response = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)
response.raise_for_status()

models = response.json()["models"]
for model in models:
    print(model["name"])


In [ ]:
!pip install dspy==3.2.1

In [ ]:
import pandas as pd
import dspy

In [ ]:
generation_df = pd.read_excel('validation_set_articulation.xlsx')

In [ ]:
generation_df

In [ ]:
generation_df.drop(columns=['Jack\'s Codes'], inplace=True)

In [ ]:
generation_df.drop(columns=['Vishal\'s Codes'], inplace=True)

In [ ]:
generation_df.drop(columns=['Notes'], inplace=True)

In [ ]:
generation_df

In [ ]:
genset = []

for context, current_turn, category in generation_df.values:
    examples = dspy.Example(context=context, question=current_turn, response=category).with_inputs("context", "question")
    genset.append(examples)

In [ ]:
genset

In [ ]:
lm = dspy.LM('ollama_chat/qwen3:4b', api_base = 'http://localhost:11434', api_key='', max_tokens=8192, cache=False, think=False)
dspy.configure(lm=lm)

In [ ]:
from typing import Literal


class TurnClassifier(dspy.Signature):
    """Given the context for a Dungeons & Dragons game turn and the game turn itself, classify the turn."""
    context: str = dspy.InputField(desc = "The three previous game turns which describe a player's action or their dialogue.")
    question: str = dspy.InputField (desc="The current turn taken by a player, which can include a description of an action or a piece of dialogue.")
    response: Literal['question game-state',
                      'assert game-state',
                      'act on game-state',
                      'propose intention or action',
                      'argue',
                      'acknowledge',
                      'probe intention',
                      'interact with resource',
                      'out-of-gameplay',
                      ] = dspy.OutputField()

class PlayerInstruction(dspy.Signature):
  # """Given the context for a game turn, the game turn itself and the category of articulation work generate a prompt to instruct an agent based on the category of articulation work."""
  # context: str = dspy.InputField(desc = "The three previous game turns which describe a player's action or their dialogue.")
  # question: str = dspy.InputField (desc="The current turn taken by a player, which can include a description of an action or a piece of dialogue.")
  category: Literal['question game-state',
                      'assert game-state',
                      'act on game-state',
                      'propose intention or action',
                      'argue',
                      'acknowledge',
                      'probe intention',
                      'interact with resource',
                      'out-of-gameplay',
                    ] = dspy.InputField()
  player_instruction:str = dspy.OutputField(desc="instruction on how to behave within a D&D game.")

In [ ]:
class ClassifyTurns(dspy.Module):
  def __init__(self):
    self.classifier = dspy.Predict(TurnClassifier, caching=False)

  def forward(self, context, question, **kwargs):
    prediction = self.classifier(context=context, question=question)
    return prediction


In [ ]:
classify = ClassifyTurns()
classify.load("optimized_classifiers/bfs_qwen3_4b_default_run_0.json")

In [ ]:
class PromptGenerator(dspy.Module):
  def __init__(self):
    self.classifier = classify
    self.generator = dspy.ChainOfThought(PlayerInstruction, caching=False)

  def forward(self, context, question, category, **kwargs):
    pred_category = classify(context=context, question=question).response
    prompt = self.generator(category=pred_category)
    return prompt, category

prompt_generator = PromptGenerator()


In [ ]:
prompt_generator(genset[5]['context'], genset[5]['question'], genset[5]['response'])

In [ ]:
new_df = generation_df.drop_duplicates('Agreed Codes')

In [ ]:
new_df

In [ ]:
new_set = []

for context, current_turn, category in new_df.values:
    examples = dspy.Example(context=context, question=current_turn, response=category).with_inputs("context", "question")
    new_set.append(examples)

In [ ]:
new_set

In [ ]:
generated_prompt = []

for i in range(len(genset)):
    generated_prompt.append(prompt_generator(genset[i]['context'], genset[i]['question'], genset[i]['response']))

In [ ]:
generated_prompt

In [ ]:
generated_prompt[1][0]['player_instruction']

In [ ]:
category = []
player_instruction = []

for i in range(len(generated_prompt)):
  category.append(generated_prompt[i][1])
  player_instruction.append(generated_prompt[i][0]['player_instruction'])


In [ ]:
category

In [ ]:
player_instruction

In [ ]:
instruction_library = list(zip(category, player_instruction))

In [ ]:
instruction_library

In [ ]:
instruction_df = pd.DataFrame(instruction_library)

In [ ]:
instruction_df.drop_duplicates(subset=[0], inplace=False)

In [ ]:
filename = 'instructions.xlsx'
instruction_df.to_excel(filename)